# Session 3, Module 07: Modules and Packages


This module covers:
- Importing modules and packages
- Creating your own modules
- The __name__ == "__main__" pattern
- Package structure and __init__.py
- Relative vs absolute imports

Data Engineering Context:
Well-organized modules and packages make ETL code
maintainable, testable, and reusable across projects.


In [ ]:
import sys
from pathlib import Path

## What Are Modules And Packages?


In [ ]:
print("=== What are Modules and Packages? ===")

print("""
MODULE: A single Python file (.py)
  - Contains functions, classes, variables
  - Can be imported by other modules
  - Example: math.py, json.py

PACKAGE: A directory containing modules
  - Contains __init__.py (can be empty)
  - Groups related modules together
  - Example: pandas/, numpy/

LIBRARY: Collection of packages/modules
  - Distributed via PyPI
  - Installed with pip
  - Example: pandas, requests, sqlalchemy
""")

## Import Statements


In [ ]:
print("\n=== Import Statements ===")

# Method 1: Import entire module
import math
print(f"math.sqrt(16) = {math.sqrt(16)}")  # OUTPUT: 4.0

# Method 2: Import specific items
from math import pi, ceil
print(f"pi = {pi:.4f}")  # OUTPUT: 3.1416
print(f"ceil(4.2) = {ceil(4.2)}")  # OUTPUT: 5

# Method 3: Import with alias
import math as m
print(f"m.floor(4.8) = {m.floor(4.8)}")  # OUTPUT: 4

from math import factorial as fact
print(f"fact(5) = {fact(5)}")  # OUTPUT: 120

Method 4: Import all (AVOID in production code)
from math import *  # Pollutes namespace, hard to track origin

## The Module Search Path


In [ ]:
print("\n=== The Module Search Path ===")

Python searches for modules in this order:
1. Current directory
2. PYTHONPATH environment variable
3. Standard library
4. Site-packages (installed packages)

In [ ]:
print("sys.path (module search locations):")
for i, path in enumerate(sys.path[:5]):
    print(f"  {i}: {path}")
if len(sys.path) > 5:
    print(f"  ... and {len(sys.path) - 5} more")

============================================================
__name__ AND __main__
============================================================

In [ ]:
print("\n=== __name__ and __main__ ===")

Every module has a __name__ attribute
When run directly: __name__ == "__main__"
When imported: __name__ == module name

In [ ]:
print(f"This module's __name__: {__name__}")

# The pattern:
print("""
# my_module.py

def main():
    '''Main function.'''
    print("Running main logic")

def helper():
    '''Helper function that can be imported.'''
    return "helper result"

This code only runs when the file is executed directly
Not when it's imported

In [ ]:
if __name__ == "__main__":
    main()
""")

# Why use this pattern?
print("""
Benefits of if __name__ == "__main__":
  1. Module can be both imported and run directly
  2. Test code doesn't run on import
  3. Makes module more reusable
  4. Standard Python convention
""")

## Package Structure


In [ ]:
print("\n=== Package Structure ===")

print("""
Example ETL package structure:

etl_pipeline/
├── __init__.py          # Makes this a package
├── extract/
│   ├── __init__.py
│   ├── api.py           # APIExtractor class
│   ├── database.py      # DatabaseExtractor class
│   └── file.py          # FileExtractor class
├── transform/
│   ├── __init__.py
│   ├── clean.py         # Cleaning functions
│   ├── validate.py      # Validation functions
│   └── enrich.py        # Enrichment functions
├── load/
│   ├── __init__.py
│   ├── warehouse.py     # WarehouseLoader class
│   └── file.py          # FileLoader class
└── utils/
    ├── __init__.py
    ├── config.py        # Configuration handling
    └── logging.py       # Logging setup
""")

============================================================
__init__.py — PACKAGE INITIALIZATION
============================================================

In [ ]:
print("\n=== __init__.py — Package Initialization ===")

print("""
__init__.py serves multiple purposes:

1. MARKS DIRECTORY AS PACKAGE
   # Empty __init__.py is fine for simple packages

2. CONTROLS PUBLIC API
   # etl_pipeline/__init__.py
   from .extract.api import APIExtractor
   from .extract.database import DatabaseExtractor
   from .transform.clean import clean_data

   __all__ = ['APIExtractor', 'DatabaseExtractor', 'clean_data']

3. PACKAGE-LEVEL INITIALIZATION
   # Run setup code when package is imported
   import logging
   logging.getLogger(__name__).addHandler(logging.NullHandler())

4. DEFINES __version__
   __version__ = "1.0.0"
""")

# Example of what __all__ does:
print("__all__ controls 'from package import *':")
print("  __all__ = ['public_func']  # Only export these")

## Relative Vs Absolute Imports


In [ ]:
print("\n=== Relative vs Absolute Imports ===")

print("""
ABSOLUTE IMPORTS (preferred in most cases):
  from etl_pipeline.extract.api import APIExtractor
  from etl_pipeline.utils.config import Config

  Pros:
    - Clear and explicit
    - Works the same everywhere
    - Easier to understand

RELATIVE IMPORTS (within packages):
  from .api import APIExtractor       # Same directory
  from ..utils.config import Config   # Parent directory
  from ...other import something      # Grandparent (rare)

  Pros:
    - Shorter when deeply nested
    - Package can be renamed without changing imports

  Cons:
    - Only work inside packages
    - Can be confusing with complex structures
""")

## Creating A Module


In [ ]:
print("\n=== Creating a Module ===")

# Let's create a simple module structure
module_example = '''
# utils.py - A utility module

"""
Utility functions for data processing.

This module provides helper functions for common
data engineering tasks.
"""

__version__ = "1.0.0"
__author__ = "Data Team"

# Constants
DEFAULT_BATCH_SIZE = 1000
VALID_STATUSES = {"active", "inactive", "pending"}


def clean_string(value: str) -> str:
    """Remove whitespace and normalize string."""
    if value is None:
        return ""
    return str(value).strip()


def validate_email(email: str) -> bool:
    """Check if email format is valid."""
    import re
    pattern = r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+$"
    return bool(re.match(pattern, email))


class DataProcessor:
    """Process data records."""

    def __init__(self, batch_size: int = DEFAULT_BATCH_SIZE):
        self.batch_size = batch_size

    def process(self, records: list) -> list:
        """Process a list of records."""
        return [self._process_one(r) for r in records]

    def _process_one(self, record: dict) -> dict:
        """Process a single record."""
        return {
            **record,
            "processed": True,
        }


# Code that runs when module is executed directly
if __name__ == "__main__":
    # Test the module
    print(f"Utils module version: {__version__}")

    # Test functions
    print(f"clean_string('  hello  '): {clean_string('  hello  ')}")
    print(f"validate_email('test@example.com'): {validate_email('test@example.com')}")

    # Test class
    processor = DataProcessor()
    result = processor.process([{"id": 1}])
    print(f"Processed: {result}")
'''

print(module_example)

## Import Best Practices


In [ ]:
print("\n=== Import Best Practices ===")

print("""
IMPORT ORDER (PEP 8):
  1. Standard library imports
  2. Related third-party imports
  3. Local application imports

  Each group separated by blank line.

  Example:
    import os
    import sys
    from pathlib import Path

    import pandas as pd
    import numpy as np

    from mypackage.utils import helper
    from mypackage.config import settings

IMPORT STYLE:
  ✓ import os                    # Standard library
  ✓ from pathlib import Path     # Specific import
  ✓ import pandas as pd          # Common alias
  ✗ from os import *             # Avoid star imports
  ✗ import os, sys, json         # One import per line

CIRCULAR IMPORTS:
  Problem: A imports B, B imports A
  Solutions:
    1. Restructure code to avoid cycle
    2. Move import inside function
    3. Use TYPE_CHECKING for type hints only

    from typing import TYPE_CHECKING
    if TYPE_CHECKING:
        from .other_module import SomeClass
""")

## Lazy Imports


In [ ]:
print("\n=== Lazy Imports ===")

print("""
Import inside function when:
  1. Module is expensive to load
  2. Module is rarely used
  3. Breaking circular imports

Example:
  def process_data(data):
      '''Process data with pandas (lazy import).'''
      import pandas as pd  # Only import when needed
      df = pd.DataFrame(data)
      return df.to_dict()

Benefits:
  - Faster startup time
  - Memory savings if function never called
  - Avoids circular import issues
""")

## Reloading Modules


In [ ]:
print("\n=== Reloading Modules ===")

print("""
During development, you might modify a module and want to reload it:

  import importlib

  # First import
  import my_module

  # After modifying my_module.py
  importlib.reload(my_module)

Note: This is mainly for interactive development.
In production, restart the process instead.
""")

## Practical Example: Project Structure


In [ ]:
print("\n=== Practical Example: Project Structure ===")

print("""
data_pipeline/
├── pyproject.toml           # Modern Python packaging
├── README.md
├── src/
│   └── data_pipeline/
│       ├── __init__.py      # Package init
│       ├── __main__.py      # python -m data_pipeline
│       ├── cli.py           # Command-line interface
│       ├── config.py        # Configuration
│       ├── extract/
│       │   ├── __init__.py
│       │   └── sources.py
│       ├── transform/
│       │   ├── __init__.py
│       │   └── processors.py
│       └── load/
│           ├── __init__.py
│           └── destinations.py
└── tests/
    ├── __init__.py
    ├── test_extract.py
    ├── test_transform.py
    └── test_load.py

__main__.py allows: python -m data_pipeline
from data_pipeline import cli
cli.main()

In [ ]:
""")

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Modules and Packages Key Points:

MODULES:
  - Single .py file
  - Import with: import module or from module import item
  - Use if __name__ == "__main__" for script entry point

PACKAGES:
  - Directory with __init__.py
  - Groups related modules
  - __init__.py controls public API

IMPORTS:
  - Absolute: from package.subpackage import item
  - Relative: from .subpackage import item (inside packages)
  - Follow PEP 8 import order

BEST PRACTICES:
  - One import per line
  - Avoid star imports
  - Use __all__ to control exports
  - Lazy imports for expensive modules

PROJECT STRUCTURE:
  - src/ layout is modern standard
  - Separate tests/ directory
  - __main__.py for python -m package
""")